# 03 Compare Baseline vs Reranked Outputs

This notebook compares the CSV artifacts produced by the first two notebooks and shows how reranking changes ordering.

## Comparison Logic
We evaluate "better" using the reranker's own objective:
- Baseline list: order from Stage 1 retrieval (`baseline_rank`).
- Reranked list: order from weighted scoring (`reranked_position`, `final_score`).

The notebook provides:
1. Side-by-side top-N lists.
2. Per-paper rank shift (`baseline_rank - reranked_position`).
3. Top-K score concentration analysis (mean `final_score` in top-K).

### Step 1: Runtime Setup
Ensure notebook imports resolve from project root.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")


Project root: /Users/juno/paper-reranking-demo


### Step 2: Load Comparison Inputs
Load baseline and reranked CSV outputs.

In [7]:
import pandas as pd
from IPython.display import HTML, display

baseline_candidates = [
    ROOT / "data" / "outputs" / "notebook_baseline_preview.csv",
    ROOT / "data" / "outputs" / "baseline_candidates.csv",
]
reranked_candidates = [
    ROOT / "data" / "outputs" / "notebook_reranked_preview.csv",
]

baseline_path = next((p for p in baseline_candidates if p.exists()), None)
reranked_path = next((p for p in reranked_candidates if p.exists()), None)

if baseline_path is None:
    raise FileNotFoundError(f"Missing baseline CSV. Checked: {baseline_candidates}")
if reranked_path is None:
    raise FileNotFoundError(f"Missing reranked CSV. Checked: {reranked_candidates}")

baseline_df = pd.read_csv(baseline_path)
reranked_df = pd.read_csv(reranked_path)

print("baseline file:", baseline_path)
print("reranked file:", reranked_path)
print("baseline rows:", len(baseline_df))
print("reranked rows:", len(reranked_df))

preview_n = min(5, len(baseline_df), len(reranked_df))

baseline_preview = baseline_df.head(preview_n)
reranked_preview = reranked_df.head(preview_n)

left_html = baseline_preview.to_html(index=False)
right_html = reranked_preview.to_html(index=False)

display(
    HTML(
        f"""
        <div style="display:flex; gap:24px; align-items:flex-start;">
          <div style="flex:1;"><h4>Baseline Preview (Top {preview_n})</h4>{left_html}</div>
          <div style="flex:1;"><h4>Reranked Preview (Top {preview_n})</h4>{right_html}</div>
        </div>
        """
    )
)


baseline file: /Users/juno/paper-reranking-demo/data/outputs/notebook_baseline_preview.csv
reranked file: /Users/juno/paper-reranking-demo/data/outputs/notebook_reranked_preview.csv
baseline rows: 10
reranked rows: 10


### Step 3: Side-by-Side Top-N View
Display baseline and reranked top entries next to each other for quick visual comparison.

In [3]:
TOP_N = min(10, len(baseline_df), len(reranked_df))

baseline_top = baseline_df.sort_values("baseline_rank").head(TOP_N)[
    ["baseline_rank", "paper_id", "title", "year"]
].rename(columns={"baseline_rank": "rank"})

reranked_top = reranked_df.sort_values("reranked_position").head(TOP_N)[
    ["reranked_position", "paper_id", "title", "year", "final_score"]
].rename(columns={"reranked_position": "rank"})

left_html = baseline_top.to_html(index=False)
right_html = reranked_top.to_html(index=False)

html = (
    '<div style="display:flex; gap:24px; align-items:flex-start;">'
    f'<div style="flex:1;"><h3>Baseline Top-{TOP_N}</h3>{left_html}</div>'
    f'<div style="flex:1;"><h3>Reranked Top-{TOP_N}</h3>{right_html}</div>'
    '</div>'
)

display(HTML(html))


rank,paper_id,title,year
1,8495965cf4ff5eb8010c7f126690106c135f23ab,Kolmogorov–Arnold graph neural networks for molecular property prediction,2025
2,ee75d0675c7adedded789e38500cc0b32b9fb4ae,Chemistry-intuitive explanation of graph neural networks for molecular property prediction with substructure masking,2023
3,c639624193847693bb2fd4b1319ca3229093c412,Mfgnn: Multi‐Scale Feature‐Attentive Graph Neural Networks for Molecular Property Prediction,2025
4,7675fc907efe86a45f3259784d644897dc95af1b,Quantitative evaluation of explainable graph neural networks for molecular property prediction,2021
5,049c6ba3c6776e474cd2b60d081f47299f85091a,Physical Pooling Functions in Graph Neural Networks for Molecular Property Prediction,2022
6,b7fdef4f00bed1ebb70d8d86534bfd76a27ffb6c,Consistency-regularized graph neural networks for molecular property prediction,2025
7,6039ef4676d535ef5b60f6315036a84b9d5ebe3b,Chain-aware graph neural networks for molecular property prediction,2024
8,be5d48ff7434525a965390f95ceb9902a46e76e0,Multi-View Graph Neural Networks for Molecular Property Prediction,2020
9,c70d99bc680615b38111f949ce1ff0cc171d3967,Cross-dependent graph neural networks for molecular property prediction,2022
10,82a4356ab71189ae9837b7752ba31fa570a04514,Composite Graph Neural Networks for Molecular Property Prediction,2024


### Step 4: Rank Shift Analysis
Compute how each paper moved after reranking.

`rank_shift = baseline_rank - reranked_position`
- positive: moved up
- negative: moved down

In [4]:
comparison_df = baseline_df.merge(
    reranked_df,
    on="paper_id",
    suffixes=("_baseline", "_reranked"),
    how="inner",
)

comparison_df["rank_shift"] = comparison_df["baseline_rank"] - comparison_df["reranked_position"]

rank_shift_view = comparison_df[
    [
        "paper_id",
        "title_reranked",
        "baseline_rank",
        "reranked_position",
        "rank_shift",
        "final_score",
        "topic_match",
        "method_match",
        "relationship_match",
        "recency",
    ]
].sort_values(["rank_shift", "final_score"], ascending=[False, False])

rank_shift_view.head(20)


,paper_id,title_reranked,baseline_rank,reranked_position,rank_shift,final_score,topic_match,method_match,relationship_match,recency
9,82a4356ab71189ae9837b7752ba31fa570a04514,Composite Graph Neural Networks for Molecular ...,10,2,8,0.687207,0.828191,0.500581,0.668663,0.8
8,c70d99bc680615b38111f949ce1ff0cc171d3967,Cross-dependent graph neural networks for mole...,9,3,6,0.679855,0.893761,0.538622,0.661810,0.4
6,6039ef4676d535ef5b60f6315036a84b9d5ebe3b,Chain-aware graph neural networks for molecula...,7,5,2,0.642465,0.721735,0.530521,0.602807,0.8
7,be5d48ff7434525a965390f95ceb9902a46e76e0,Multi-View Graph Neural Networks for Molecular...,8,7,1,0.591637,0.732764,0.585292,0.638329,0.0
0,8495965cf4ff5eb8010c7f126690106c135f23ab,Kolmogorov–Arnold graph neural networks for mo...,1,1,0,0.710547,0.896650,0.460850,0.633858,1.0
5,b7fdef4f00bed1ebb70d8d86534bfd76a27ffb6c,Consistency-regularized graph neural networks ...,6,6,0,0.595652,0.694136,0.431453,0.493074,1.0
2,c639624193847693bb2fd4b1319ca3229093c412,Mfgnn: Multi‐Scale Feature‐Attentive Graph Neu...,3,4,-1,0.647538,0.702769,0.470157,0.642085,1.0
4,049c6ba3c6776e474cd2b60d081f47299f85091a,Physical Pooling Functions in Graph Neural Net...,5,8,-3,0.588011,0.776253,0.412866,0.609851,0.4
3,7675fc907efe86a45f3259784d644897dc95af1b,Quantitative evaluation of explainable graph n...,4,10,-6,0.548869,0.725634,0.427718,0.586328,0.2
1,ee75d0675c7adedded789e38500cc0b32b9fb4ae,Chemistry-intuitive explanation of graph neura...,2,9,-7,0.573902,0.711176,0.400426,0.579453,0.6


### Step 5: "Better" Summary Under the Reranker Objective
Compare mean final scores for top-K papers chosen by baseline vs reranked ordering.

Higher reranked top-K mean indicates stronger concentration of high-scoring papers near the top.

In [5]:
def topk_mean_final_for_baseline(k: int) -> float:
    ids = baseline_df.nsmallest(k, "baseline_rank")["paper_id"]
    return float(reranked_df[reranked_df["paper_id"].isin(ids)]["final_score"].mean())


def topk_mean_final_for_reranked(k: int) -> float:
    return float(reranked_df.nsmallest(k, "reranked_position")["final_score"].mean())


max_k = min(len(baseline_df), len(reranked_df))
k_values = sorted(set([k for k in [3, 5, 10] if k <= max_k]))
if not k_values and max_k > 0:
    k_values = [max_k]

rows = []
for k in k_values:
    baseline_mean = topk_mean_final_for_baseline(k)
    reranked_mean = topk_mean_final_for_reranked(k)
    rows.append(
        {
            "top_k": k,
            "baseline_topk_mean_final_score": baseline_mean,
            "reranked_topk_mean_final_score": reranked_mean,
            "absolute_lift": reranked_mean - baseline_mean,
        }
    )

summary_df = pd.DataFrame(rows)
summary_df


,top_k,baseline_topk_mean_final_score,reranked_topk_mean_final_score,absolute_lift
0,3,0.643996,0.692536,0.048541
1,5,0.613774,0.673522,0.059749
2,10,0.626568,0.626568,0.000000


### Step 6: Save Comparison Artifacts
Persist rank-shift and summary tables for reporting.

In [6]:
out_rank_shift = ROOT / "data" / "outputs" / "comparison_rank_shift.csv"
out_summary = ROOT / "data" / "outputs" / "comparison_summary.csv"

rank_shift_view.to_csv(out_rank_shift, index=False)
summary_df.to_csv(out_summary, index=False)

print(f"Saved: {out_rank_shift}")
print(f"Saved: {out_summary}")


Saved: /Users/juno/paper-reranking-demo/data/outputs/comparison_rank_shift.csv
Saved: /Users/juno/paper-reranking-demo/data/outputs/comparison_summary.csv
